In [ ]:
# Celda 1
# Automatic control of the final model: runs after 14_materiales and flags what needs work, so that nobody has to
# look for problems by hand. Writes data/control/informe_control.csv and renders/control_cubiertas.png.
%cd ..

MODEL_JSON = "data/colegio_imperial_meshes.json"     # Final meshes with materials (14_materiales)
BLEND_FILE = "blender/colegio_imperial_lod3.blend"
GLB_FILE = "blender/colegio_imperial_lod3.glb"
CLOUD = "data/nube_manzana_2023.laz"
GPKG = "data/catastro_manzana.gpkg"
REPORT = "data/control/informe_control.csv"
OPENINGS_REPORT = "data/control/derrames.csv"   # 17_detalles
ROOF_IMAGE = "renders/control_cubiertas.png"
ORIGIN = (440000.0, 4473700.0, 640.0)
CONJUNTO = ["0139402VK4703G", "0139424VK4703G", "0139423VK4703G"]

# Thresholds:
CELL = 0.5                 # Grid of the roof comparison (m)
ROOF_OK_SHARE = 0.70       # Share of cells of a part within ROOF_TOL of the cloud
ROOF_TOL = 0.5
ROOF_MAX_MEDIAN = 0.3      # |median difference| of a part (m)
BLANK_WALL_AREA = 50.0     # Outer walls larger than this (m2) must have openings
ALLOWED_GRID = ["0139402VK4703G_part6", "0139402VK4703G_part9", "0139402VK4703G_part12"]   # Prisms under towers / dome
STEEP_TOL = 1.0            # Towers and dome: steep slate; on 0.5 m cells a 60 degree slope changes 0.85 m per cell

In [ ]:
# Celda 2
import os
import re
import csv
import json
import math
import numpy as np
import laspy
import shapely
import geopandas as gpd
from shapely.geometry import Point
from PIL import Image

os.makedirs(os.path.dirname(REPORT), exist_ok=True)
with open(MODEL_JSON, encoding="utf-8") as f:
    meshes = json.load(f)["parts"]
parts = gpd.read_file(GPKG, layer="building_parts")
parts["geometry"] = parts.geometry.translate(-ORIGIN[0], -ORIGIN[1])
results = []   # (check, object, value, threshold, status)

def record(check, obj, value, threshold, ok):
    results.append({"control": check, "objeto": obj, "valor": value, "umbral": threshold, "estado": "OK" if ok else "AVISO"})
print(len(meshes), "meshes")

In [ ]:
# Celda 3
# 1. Files: the model was saved after its input, and the glb holds the same objects.
model_time = os.path.getmtime(MODEL_JSON)
for path in (BLEND_FILE, GLB_FILE):
    fresh = os.path.exists(path) and os.path.getmtime(path) >= model_time
    record("archivo al día", path, round(os.path.getsize(path) / 1e6, 2) if os.path.exists(path) else 0, "más nuevo que " + MODEL_JSON, fresh)

# 2. Materials: every mesh of the ensemble has a texture or a color.
ensemble = [m for m in meshes if m.get("parcel") in CONJUNTO or m.get("group") in ("piedra", "vidrio", "metal", "pizarra", "volumen")]
missing = [m["name"] for m in ensemble if "texture" not in m and "color" not in m]
record("materiales", "conjunto", len(missing), "0 piezas sin material", not missing)

# 3. Grid leftovers: LOD2 parts of the ensemble still drawn with the 0.5 m grid (not bays, not allowed prisms).
def base_name(name):
    return name.replace("_cubierta", "").replace("_escalones", "").replace("_huecos", "")
grid = sorted({base_name(m["name"]) for m in meshes
               if m.get("lod") == 2 and m.get("parcel") in CONJUNTO and "_c" not in base_name(m["name"]).split("_part")[-1]})
grid = [g for g in grid if g not in ALLOWED_GRID]
for name in grid:
    record("grilla sobrante", name, "grilla 0,5 m", "cubierta por crujías", False)
if not grid:
    record("grilla sobrante", "conjunto", 0, "0 partes", True)

In [ ]:
# Celda 4
# 4. Roofs against the cloud: model height (highest surface) and cloud height (p90 of the building points) on a 0.5 m
# grid over the ensemble; per Catastro part, the share of cells within ROOF_TOL and the median difference.
footprint = shapely.union_all(parts[parts["localId"].str[:14].isin(CONJUNTO)].geometry)
x0, y0, x1, y1 = footprint.bounds
ncol, nrow = int((x1 - x0) / CELL) + 1, int((y1 - y0) / CELL) + 1
model = np.full((nrow, ncol), np.nan)
gx = x0 + (np.arange(ncol) + 0.5) * CELL
gy = y0 + (np.arange(nrow) + 0.5) * CELL
for m in meshes:
    if m.get("parcel") not in CONJUNTO and m.get("group") != "volumen":
        continue
    v = np.array(m["vertices"])
    for face in m["faces"]:
        for k in range(1, len(face) - 1):                      # Fan triangles
            (ax, ay, az), (bx, by, bz), (cx, cy, cz) = v[face[0]], v[face[k]], v[face[k + 1]]
            det = (by - cy) * (ax - cx) + (cx - bx) * (ay - cy)
            if abs(det) < 1e-9:
                continue                                         # Vertical triangle: no area seen from above
            c0, c1 = np.searchsorted(gx, min(ax, bx, cx)), np.searchsorted(gx, max(ax, bx, cx))
            r0, r1 = np.searchsorted(gy, min(ay, by, cy)), np.searchsorted(gy, max(ay, by, cy))
            if c1 <= c0 or r1 <= r0:
                continue
            X, Y = np.meshgrid(gx[c0:c1], gy[r0:r1])
            l1 = ((by - cy) * (X - cx) + (cx - bx) * (Y - cy)) / det
            l2 = ((cy - ay) * (X - cx) + (ax - cx) * (Y - cy)) / det
            l3 = 1 - l1 - l2
            inside = (l1 >= 0) & (l2 >= 0) & (l3 >= 0)
            Z = l1 * az + l2 * bz + l3 * cz + ORIGIN[2]
            block = model[r0:r1, c0:c1]
            block[inside] = np.fmax(block[inside], Z[inside])

# Cloud height of a cell: 90th percentile of its building points (the maximum catches chimneys and antennas).
import pandas as pd
keys, heights = [], []
with laspy.open(CLOUD) as reader:
    for points in reader.chunk_iterator(1_000_000):
        b = np.array(points.classification) == 6
        px, py, pz = np.array(points.x)[b] - ORIGIN[0], np.array(points.y)[b] - ORIGIN[1], np.array(points.z)[b]
        c = ((px - x0) / CELL).astype(int)
        r = ((py - y0) / CELL).astype(int)
        ok = (c >= 0) & (c < ncol) & (r >= 0) & (r < nrow)
        keys.append(r[ok] * ncol + c[ok])
        heights.append(pz[ok])
p90 = pd.Series(np.concatenate(heights)).groupby(np.concatenate(keys)).quantile(0.9)
cloud_top = np.full(nrow * ncol, np.nan)
cloud_top[p90.index.values] = p90.values
cloud_top = cloud_top.reshape(nrow, ncol)
del keys, heights

diff = model - cloud_top
X, Y = np.meshgrid(gx, gy)
for _, row in parts[parts["localId"].str[:14].isin(CONJUNTO)].iterrows():
    inside = shapely.contains_xy(row.geometry.buffer(-0.5), X, Y) & ~np.isnan(diff)
    if inside.sum() < 20:
        continue
    d = diff[inside]
    tol = STEEP_TOL if row["localId"] in ALLOWED_GRID else ROOF_TOL
    share = float(np.mean(np.abs(d) < tol))
    median = float(np.median(d))
    record("cubierta contra nube", row["localId"], f"{share:.0%} dentro de ±{tol} m; mediana {median:+.2f} m",
           f"≥ {ROOF_OK_SHARE:.0%}; |mediana| ≤ {ROOF_MAX_MEDIAN} m", share >= ROOF_OK_SHARE and abs(median) <= ROOF_MAX_MEDIAN)

# Picture: blue = model below the cloud, red = above, white = within ±ROOF_TOL; grey = no data.
scaled = np.clip(diff / 2.0, -1, 1)
rgb = np.full((nrow, ncol, 3), 200.0)
valid = ~np.isnan(scaled)
rgb[valid] = 255
neg = valid & (scaled < 0)
pos = valid & (scaled > 0)
rgb[neg, 0] = rgb[neg, 1] = 255 * (1 + scaled[neg])
rgb[pos, 1] = rgb[pos, 2] = 255 * (1 - scaled[pos])
Image.fromarray(np.flipud(rgb).astype(np.uint8)).resize((ncol * 4, nrow * 4), Image.NEAREST).save(ROOF_IMAGE)
print("Roof map:", ROOF_IMAGE)

In [ ]:
# Celda 5
# 5. Blank facades: outer walls of the ensemble (on the border of the union of all footprints: streets and
# courtyards, not party walls) larger than BLANK_WALL_AREA with no opening in front of them.
outline = shapely.union_all(parts.geometry).boundary
openings = [m for m in meshes if m.get("lod") == 3 and (m.get("material") in ("ventana", "sombra") or "hueco" in m["name"] or "arco" in m["name"])]
opening_points = np.array([np.mean(m["vertices"], axis=0)[:2] for m in openings]) if openings else np.zeros((0, 2))
walls = {}
for m in meshes:
    if m.get("lod") != 2 or m.get("parcel") not in CONJUNTO:
        continue
    v = m["vertices"]
    for face in m["faces"]:
        if len(face) != 4:
            continue
        (ax, ay, az), (cx, cy, cz), (_, _, ct), (_, _, at) = [v[i] for i in face]
        length = math.hypot(cx - ax, cy - ay)
        if length < 0.4 or abs(az - cz) > 0.01:
            continue
        if outline.distance(Point((ax + cx) / 2, (ay + cy) / 2)) > 0.3:
            continue
        t = np.array([cx - ax, cy - ay]) / length
        key = (base_name(m["name"]).split("_c")[0], round(math.degrees(math.atan2(t[1], t[0])) / 5) * 5,
               round((ax * -t[1] + ay * t[0]) / 0.5) * 0.5)        # Same part, direction and line
        w = walls.setdefault(key, {"area": 0.0, "points": [], "t": t})
        w["area"] += length * max((at + ct) / 2 - max(az, cz) - 1.0, 0)   # Minus the 1 m sunk into the ground
        w["points"] += [(ax, ay), (cx, cy)]
blank = 0
for (name, angle, offset), w in walls.items():
    if w["area"] < BLANK_WALL_AREA:
        continue
    pts = np.array(w["points"])
    t = w["t"]
    n = np.array([-t[1], t[0]])
    s = pts @ t
    if len(opening_points):
        os_ = opening_points @ t
        on = np.abs(opening_points @ n - (pts @ n).mean()) < 1.5
        has = (on & (os_ >= s.min()) & (os_ <= s.max())).any()
    else:
        has = False
    if not has:
        blank += 1
        record("fachada ciega", f"{name} (dirección {angle}°)", f"{w['area']:.0f} m2 sin huecos", f"huecos en muros > {BLANK_WALL_AREA:.0f} m2", False)
if not blank:
    record("fachada ciega", "conjunto", 0, f"muros > {BLANK_WALL_AREA:.0f} m2 con huecos", True)

In [ ]:
# Celda 5b
# 6. Openings without depth: every opening must have found its wall in 17_detalles (data/control/derrames.csv).
with open(OPENINGS_REPORT, encoding="utf-8") as f:
    reveals = list(csv.DictReader(f))
flat = [r["hueco"] for r in reveals if r["estado"] == "sin muro detrás"]
record("huecos con derrame", "conjunto", f"{sum(r['estado'] == 'OK' for r in reveals)} de {len(reveals) - sum(r['estado'].startswith('eliminado') for r in reveals)}", "todos con muro detrás", not flat)
for name in flat:
    record("hueco sin derrame", name, "sin muro detrás", "muro a menos de 0,5 m", False)

# 7. Abstract volumes: rows of the tables still drawn as a box instead of their shape ("simplificado a caja",
# "como caja", "como cuadrado" in the origin column).
import glob
boxes = []
for path in sorted(glob.glob("data/fachadas/*.csv")) + ["data/volumenes/huecos.csv"]:
    with open(path, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            if re.search(r"(simplificad[oa] a caja|como caja|como cuadrado)", row.get("origen", "")):
                boxes.append((os.path.basename(path), row.get("elemento") or row.get("pieza")))
for table, name in boxes:
    record("volumen abstracto", f"{table}: {name}", "caja", "forma leída en la foto", False)
if not boxes:
    record("volumen abstracto", "tablas", 0, "0 filas simplificadas a caja", True)

# Celda 6
# Report:
import csv
with open(REPORT, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(results[0]))
    writer.writeheader()
    writer.writerows(results)
warnings = [r for r in results if r["estado"] == "AVISO"]
print(len(results), "controls |", len(results) - len(warnings), "OK |", len(warnings), "AVISO")
for r in warnings:
    print("AVISO", r["control"], "|", r["objeto"], "|", r["valor"])